In [1]:
# ===============================
# CRISP-DM | Preparación de datos
# Paso: Carga de datos limpios diarios
# ===============================

import pandas as pd
from google.colab import files

# Subir archivo desde el equipo
uploaded = files.upload()

# Nombre del archivo (debe coincidir exactamente)
file_name = "datos_limpios_diarios_cacao_crispdm_preparacion.xlsx"

# Leer archivo Excel
df = pd.read_excel(file_name)

# Mostrar información general
df.info()

# Mostrar las primeras filas
df.head()


Saving datos_limpios_diarios_cacao_crispdm_preparacion.xlsx to datos_limpios_diarios_cacao_crispdm_preparacion.xlsx
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5293 entries, 0 to 5292
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Fecha          5293 non-null   datetime64[ns]
 1   Mes            5293 non-null   int64         
 2   Año            5293 non-null   int64         
 3   Día            5293 non-null   int64         
 4   PRECIPITACIÓN  5293 non-null   float64       
 5   TMAX           5293 non-null   float64       
 6   TMIN           5293 non-null   float64       
 7   TPROM          5293 non-null   float64       
 8   HR             5293 non-null   float64       
 9   RAD_SOL        5293 non-null   float64       
 10  EVPTRNS        5293 non-null   float64       
 11  VDL            5293 non-null   float64       
dtypes: datetime64[ns](1), float64(8), int64(3)
memory usage: 4

,Fecha,Mes,Año,Día,PRECIPITACIÓN,TMAX,TMIN,TPROM,HR,RAD_SOL,EVPTRNS,VDL
0,2010-01-01,1,2010,1,0.0,27.51,10.47,18.990,61.57,21.92,3.93,0.843932
1,2010-01-02,1,2010,2,0.0,28.04,12.77,20.405,58.46,21.59,3.33,0.995942
2,2010-01-03,1,2010,3,0.0,28.26,10.05,19.155,54.09,22.92,2.92,1.018620
3,2010-01-04,1,2010,4,0.0,27.82,12.38,20.100,50.96,21.90,2.43,1.153811
4,2010-01-05,1,2010,5,0.0,28.90,13.38,21.140,54.18,22.44,2.22,1.149382


## Documentación del proceso – Preparación y transformación de datos climáticos
# Metodología CRISP-DM
🔹 Descripción general del proceso ejecutado

En esta fase del proyecto se desarrolló la preparación y transformación de los datos climáticos, siguiendo la fase Data Preparation de la metodología CRISP-DM, con el objetivo de construir un conjunto de datos mensual, consistente y agronómicamente interpretable, que sirva como insumo para el análisis y modelado del cultivo de cacao en el municipio de San Vicente de Chucurí (Santander).

El proceso incluyó la validación de calidad, selección de variables relevantes, definición de reglas físicas y la agregación temporal de datos diarios a escala mensual.

| Etapa          | Actividad                        | Descripción                                                                             |
| -------------- | -------------------------------- | --------------------------------------------------------------------------------------- |
| Carga de datos | Lectura de dataset limpio diario | Se cargó el archivo `datos_limpios_diarios_cacao_crispdm_preparacion.xlsx`              |
| Validación     | Revisión de estructura y tipos   | Se verificó ausencia de valores nulos y tipos de datos correctos                        |
| Selección      | Variables climáticas             | Se trabajó con precipitación, temperatura, humedad, radiación, evapotranspiración y VDL |
| Transformación | Agregación mensual               | Se aplicaron reglas de suma o promedio según el significado físico                      |
| Resultado      | Dataset mensual                  | Se obtuvo una base con 174 registros mensuales (2010–2024)                              |


# Variables climáticas utilizadas y rangos óptimos para el cacao

A continuación se describen las variables climáticas seleccionadas, junto con sus rangos óptimos agronómicos, los cuales sustentan su inclusión en el modelo.


| Variable                   | Descripción                            | Rango óptimo para cacao |
| -------------------------- | -------------------------------------- | ----------------------- |
| **PRECIPITACIÓN (mm/mes)** | Acumulado mensual de lluvia            | 100 – 250 mm            |
| **TMAX (°C)**              | Temperatura máxima promedio mensual    | ≤ 32 °C                 |
| **TMIN (°C)**              | Temperatura mínima promedio mensual    | ≥ 18 °C                 |
| **TPROM (°C)**             | Temperatura media mensual              | 22 – 28 °C              |
| **HR (%)**                 | Humedad relativa promedio mensual      | 70 – 90 %               |
| **RAD_SOL (MJ/m²/día)**    | Radiación solar media mensual          | 16 – 22                 |
| **EVPTRNS (mm/mes)**       | Evapotranspiración potencial acumulada | < Precipitación         |
| **VDL (kPa)**              | Déficit de presión de vapor            | < 1.2 kPa               |


In [2]:
# ===============================
# CRISP-DM | Preparación de datos
# Paso: Transformación mensual
# ===============================

# Crear columna Año-Mes
df["AÑO_MES"] = df["Fecha"].dt.to_period("M")

# Agregación mensual según criterios definidos
df_mensual = (
    df.groupby("AÑO_MES")
    .agg({
        "PRECIPITACIÓN": "sum",
        "TMAX": "mean",
        "TMIN": "mean",
        "TPROM": "mean",
        "HR": "mean",
        "RAD_SOL": "mean",
        "EVPTRNS": "sum",
        "VDL": "mean"
    })
    .reset_index()
)

# Convertir AÑO_MES a fecha (primer día del mes)
df_mensual["Fecha"] = df_mensual["AÑO_MES"].dt.to_timestamp()

# Reordenar columnas
df_mensual = df_mensual[[
    "Fecha",
    "PRECIPITACIÓN",
    "TMAX",
    "TMIN",
    "TPROM",
    "HR",
    "RAD_SOL",
    "EVPTRNS",
    "VDL"
]]

# Vista general
df_mensual.info()
df_mensual.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174 entries, 0 to 173
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Fecha          174 non-null    datetime64[ns]
 1   PRECIPITACIÓN  174 non-null    float64       
 2   TMAX           174 non-null    float64       
 3   TMIN           174 non-null    float64       
 4   TPROM          174 non-null    float64       
 5   HR             174 non-null    float64       
 6   RAD_SOL        174 non-null    float64       
 7   EVPTRNS        174 non-null    float64       
 8   VDL            174 non-null    float64       
dtypes: datetime64[ns](1), float64(8)
memory usage: 12.4 KB


,Fecha,PRECIPITACIÓN,TMAX,TMIN,TPROM,HR,RAD_SOL,EVPTRNS,VDL
0,2010-01-01,5.0,29.514194,14.937742,22.225968,58.620645,20.275806,34.15,1.113695
1,2010-02-01,225.0,30.565714,17.261786,23.913750,63.353571,18.280714,5.99,1.099789
2,2010-03-01,108.0,29.301613,17.128710,23.215161,69.386774,17.742258,19.56,0.883210
3,2010-04-01,130.0,26.645333,17.449667,22.047500,77.794000,16.946667,27.47,0.607288
4,2010-05-01,347.0,24.838387,16.778065,20.808226,82.439677,18.116452,98.70,0.444715


In [3]:
# ===============================
# CRISP-DM | Preparación de datos
# Paso: Exportar dataset mensual
# ===============================

from google.colab import files

# Nombre base del archivo
file_base = "datos_mensuales_cacao_crispdm_preparacion"

# Exportar a Excel
excel_path = f"{file_base}.xlsx"
df_mensual.to_excel(excel_path, index=False)

# Exportar a CSV
csv_path = f"{file_base}.csv"
df_mensual.to_csv(csv_path, index=False)

# Descargar archivos
files.download(excel_path)
files.download(csv_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>